# Modelado y evaluación


**Antes de ejecutar:** `python src/preprocess.py` (genera `data/processed/`, que no se sube a Git).

### 0. Preparación en Google Colab (omitir si trabajas en local con el repo clonado)

En Colab el notebook corre aislado: **no tiene** `src/preprocess.py` ni los datos. Esta celda clona el repo, entra en su carpeta y ejecuta `preprocess.py` para generar `data/processed/`.


In [1]:
import os, subprocess, sys

if "google.colab" in sys.modules and not os.path.exists("src/preprocess.py"):
    REPO_URL = "https://github.com/Bootcamp-DA-P3/Proyecto_8_Clasificacion_Lena_Tania_Franco.git"
    BRANCH = "dev"   # cambia a "main" si preprocess.py aún no está en dev
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "repo_p8"], check=True)
    os.chdir("repo_p8")

# Generar data/processed/ si aún no existe (requiere el CSV en data/raw/)
if os.path.exists("src/preprocess.py") and not os.path.exists("data/processed/train_raw.csv"):
    subprocess.run([sys.executable, "src/preprocess.py"], check=True)

print("Directorio de trabajo:", os.getcwd())
print("preprocess.py:", os.path.exists("src/preprocess.py"),
      "| train_raw.csv:", os.path.exists("data/processed/train_raw.csv"))

Directorio de trabajo: /content/repo_p8
preprocess.py: True | train_raw.csv: True


## 1. Configuración e importaciones

Aquí solo preparamos el entorno. Qué hace cada bloque:

- **Detección de la raíz del repo.** El notebook vive en `notebooks/`, pero los datos están en `data/` y el código de Preprocessing en `src/`. Si el directorio actual se llama `notebooks`, subimos un nivel. Después añadimos la raíz a `sys.path` para poder hacer `from src.preprocess import ...`.
- **Importamos del preprocesado** las listas de columnas (`NUM_FEATURES`, `BIN_FEATURES`, `CAT_FEATURES`), el nombre del target, `RANDOM_STATE=42` y la función `construir_preprocesador()`.
- **Constantes del equipo (Fase 0):**
  - `POS_LABEL = 0`: la clase que nos interesa detectar es *"sin cuenta"* (excluido financiero). Es la minoritaria (~26%) y la que le importa a negocio, por eso todas las métricas de precision/recall/F1 se calculan sobre ella.
  - `MAX_GAP = 0.05`: límite de overfitting (diferencia de F1 entre train y test).
- **`MUESTRA`**: si pones un número (p. ej. `30000`), la validación cruzada y el tuning usan solo esa cantidad de filas de train para ir rápido mientras pruebas. Déjalo en `None` para la ejecución final.
- **LightGBM / XGBoost son opcionales**: si no están instalados, simplemente se omiten.

In [2]:
import sys, json, time, warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform, randint, uniform

from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score,
                             brier_score_loss, confusion_matrix, f1_score, make_scorer,
                             precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_validate)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

# --- raíz del repo e importación del preprocesado de Franco ---
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocess import (NUM_FEATURES, BIN_FEATURES, CAT_FEATURES, TARGET, RANDOM_STATE,
                            PROCESSED_DIR, construir_preprocesador)

# --- convenciones del equipo ---
FEATURES = NUM_FEATURES + BIN_FEATURES + CAT_FEATURES
POS_LABEL = 0          # clase de interés: SIN cuenta
MAX_GAP = 0.05         # overfitting máximo permitido (F1 train - F1 test)
MUESTRA = None         # p. ej. 30000 para pruebas rápidas de CV y tuning
# --- tuning (la parte más lenta del notebook) ---
EJECUTAR_TUNING = True   # False = saltar el tuning y usar los hiperparámetros por defecto
N_ITER_TUNING = 15       # configuraciones que prueba (25 = más fino, más lento)
TUNING_FOLDS = 3         # folds de la búsqueda (5 = más fino, más lento)
TUNING_MUESTRA = 40000   # filas de train usadas SOLO para buscar (None = todas)

# --- carpetas de salida ---
TABLES_DIR = ROOT / "reports" / "tables"
FIGURES_DIR = ROOT / "reports" / "figures"
MODELS_DIR = ROOT / "models"
for d in (TABLES_DIR, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore", category=UserWarning)
print("Raíz del repo:", ROOT)
print("Features:", FEATURES)

Raíz del repo: /content/repo_p8
Features: ['age', 'educ', 'inc_q', 'female', 'emp_in', 'urban', 'internet_use', 'regionwb', 'economycode']


## 2. Carga de datos

`preprocess.py` genera **dos versiones** del split:

| Archivo | Contenido | ¿Lo usamos? |
|---|---|---|
| `train_raw.csv` / `test_raw.csv` | Datos limpios **sin codificar** (female/emp_in/urban ya en 0/1, regionwb y economycode aún como texto) + `account` | **Sí** |
| `train.csv` / `test.csv` | Ya codificados y escalados | No (solo para inspección) |

**¿Por qué usamos los `_raw` y no los ya codificados?** Porque metemos el preprocesador **dentro** de un `Pipeline`. Así, en la validación cruzada el imputador, el escalador y el one-hot se ajustan **solo con las filas de entrenamiento de cada fold**. Si usáramos los datos ya codificados, el escalador habría "visto" las filas de validación → fuga de información (*data leakage*) y métricas optimistas.

La división 80/20 estratificada con `random_state=42` ya está hecha en preprocess. El test se usa **una sola vez** al final para medir y no para decidir nada.

Lo que verificamos en la celda: tamaños, proporción de clases (esperamos ~74% con cuenta / ~26% sin cuenta) y nulos por columna (esperamos pocos, sobre todo en `educ`, `inc_q`, `emp_in`, `urban`; los maneja el imputador del pipeline).

In [3]:
def cargar_split():
    """Lee train_raw.csv y test_raw.csv y devuelve X_train, y_train, X_test, y_test."""
    tr, te = PROCESSED_DIR / "train_raw.csv", PROCESSED_DIR / "test_raw.csv"
    if not (tr.exists() and te.exists()):
        raise FileNotFoundError("Faltan train_raw.csv / test_raw.csv. "
                                "Ejecuta primero: python src/preprocess.py")
    train, test = pd.read_csv(tr), pd.read_csv(te)
    return train[FEATURES], train[TARGET], test[FEATURES], test[TARGET]


X_train, y_train, X_test, y_test = cargar_split()

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print("\nProporción de clases en train (0 = sin cuenta, 1 = con cuenta):")
print(y_train.value_counts(normalize=True).round(3).to_string())
print("\nNulos por columna en train:")
print(X_train.isna().mean().round(4).to_string())
X_train.head()

X_train: (115272, 9) | X_test: (28818, 9)

Proporción de clases en train (0 = sin cuenta, 1 = con cuenta):
account
1    0.738
0    0.262

Nulos por columna en train:
age             0.0000
educ            0.0041
inc_q           0.0068
female          0.0000
emp_in          0.0277
urban           0.0173
internet_use    0.0000
regionwb        0.0000
economycode     0.0000


,age,educ,inc_q,female,emp_in,urban,internet_use,regionwb,economycode
0,27,2.0,5.0,0,0.0,0.0,1,Middle East & North Africa (excluding high inc...,EGY
1,43,3.0,1.0,1,1.0,1.0,1,East Asia & Pacific (excluding high income),IDN
2,30,2.0,2.0,0,0.0,0.0,1,Sub-Saharan Africa (excluding high income),MOZ
3,29,3.0,3.0,0,1.0,NaN,1,High income,PRT
4,19,2.0,4.0,1,0.0,0.0,1,Middle East & North Africa (excluding high inc...,LBY


## 3. Funciones auxiliares y Baseline

Antes del baseline definimos las piezas que usaremos en **todo** el notebook. Es la parte más importante para entender el resto.

3.1 `crear_pipeline(clf)`
Un `Pipeline` encadena dos pasos: **(1)** `prep` = el `ColumnTransformer` (imputa nulos, escala numéricas, one-hot de categóricas) y **(2)** `modelo` = el clasificador. Al llamar `.fit(X, y)` el pipeline ajusta primero el preprocesador y luego el modelo; al llamar `.predict_proba(X)` aplica ambos. Esto garantiza que lo que entrenamos es exactamente lo que usará la app.

3.2 `p_sin_cuenta(pipe, X)`
`predict_proba` devuelve dos columnas ordenadas según `pipe.classes_` (que es `[0, 1]`). Buscamos la posición de la clase 0 y devolvemos esa columna: **la probabilidad de que la persona NO tenga cuenta**. Es la probabilidad que le importa a negocio.

3.3 `predecir_clase(p0, umbral)`
Convierte probabilidad en clase: si `P(sin cuenta) ≥ umbral` → 0 (sin cuenta), si no → 1 (con cuenta). Con el umbral por defecto 0.5 es lo normal; más adelante (8) ajustaremos el umbral para equilibrar precision y recall.

3.4 `metricas(y_true, p0, umbral)`
Calcula todo lo que pide el informe. Cómo leerlas, siempre referidas a **"sin cuenta"**:

- **accuracy**: % de aciertos totales. *Engañosa con desbalance*: un modelo que dijera "todos tienen cuenta" tendría ~74%.
- **precision_sin_cuenta**: de las personas que el modelo marca como "sin cuenta", ¿qué % lo es realmente? (Baja precision = campañas de inclusión mal dirigidas.)
- **recall_sin_cuenta**: de las personas realmente "sin cuenta", ¿qué % detecta el modelo? (Baja recall = excluidos que se nos escapan.)
- **f1_sin_cuenta**: media armónica de precision y recall. **Es nuestra métrica principal** y la que usamos para medir overfitting.
- **f1_macro**: promedio del F1 de ambas clases.
- **roc_auc**: capacidad de *ordenar* (¿puntúa más alto a los excluidos que a los bancarizados?). No depende del umbral. 0.5 = azar, 1 = perfecto. Para calcularla ponemos `y_excl = 1` si la persona es "sin cuenta" y usamos `p0` como puntuación.
- **pr_auc** (average precision): resumen de la curva precision-recall; más informativa que ROC cuando la clase de interés es minoritaria.
- **brier**: error cuadrático de las probabilidades (menor = probabilidades mejor calibradas).

3.5 `evaluar_overfitting(...)`
Calcula las métricas en train y en test y la diferencia: **`gap = F1_train − F1_test`**. Si `gap < 0.05` → cumple el requisito del README de Factoría (overfitting < 5%).
Intuición: un modelo que "memoriza" el train saca F1 alto ahí y bajo en test → gap grande.

3.6 `mejor_umbral_f1(y_true, p0)`
Recorre todos los umbrales posibles con `precision_recall_curve` y devuelve el que maximiza el F1 de "sin cuenta". Se usará solo con predicciones del **train** (§8) para no "espiar" el test.

In [4]:
def crear_pipeline(clf):
    """Preprocesador + clasificador. Se re-ajusta en cada fold de la CV (sin fuga)."""
    return Pipeline([("prep", construir_preprocesador()), ("modelo", clf)])


def p_sin_cuenta(pipe, X):
    """Probabilidad de la clase 0 (sin cuenta)."""
    return pipe.predict_proba(X)[:, list(pipe.classes_).index(POS_LABEL)]


def predecir_clase(p0, umbral=0.5):
    """0 = sin cuenta si P(sin cuenta) >= umbral; 1 = con cuenta."""
    return np.where(np.asarray(p0) >= umbral, 0, 1)


def metricas(y_true, p0, umbral=0.5):
    y_true = np.asarray(y_true)
    y_pred = predecir_clase(p0, umbral)
    y_excl = (y_true == POS_LABEL).astype(int)      # 1 = sin cuenta (para ROC y PR)
    return {
        "umbral": float(umbral),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_sin_cuenta": precision_score(y_true, y_pred, pos_label=POS_LABEL, zero_division=0),
        "recall_sin_cuenta": recall_score(y_true, y_pred, pos_label=POS_LABEL, zero_division=0),
        "f1_sin_cuenta": f1_score(y_true, y_pred, pos_label=POS_LABEL, zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y_excl, p0),
        "pr_auc": average_precision_score(y_excl, p0),
        "brier": brier_score_loss(y_excl, p0),
    }


def evaluar_overfitting(pipe, Xtr, ytr, Xte, yte, umbral=0.5):
    """Métricas en train y test + gap de F1 (criterio del equipo: gap < 0.05)."""
    m_tr = metricas(ytr, p_sin_cuenta(pipe, Xtr), umbral)
    m_te = metricas(yte, p_sin_cuenta(pipe, Xte), umbral)
    gap = m_tr["f1_sin_cuenta"] - m_te["f1_sin_cuenta"]
    return {"train": m_tr, "test": m_te, "gap_f1": gap,
            "gap_accuracy": m_tr["accuracy"] - m_te["accuracy"],
            "overfitting_ok": bool(gap < MAX_GAP)}


def mejor_umbral_f1(y_true, p0):
    """Umbral que maximiza el F1 de 'sin cuenta' (usar solo con predicciones de TRAIN)."""
    y_excl = (np.asarray(y_true) == POS_LABEL).astype(int)
    prec, rec, thr = precision_recall_curve(y_excl, p0)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    return float(thr[int(np.argmax(f1))])


def guardar_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=float, ensure_ascii=False))


# scorers para validación cruzada (mismas métricas, con pos_label=0)
SCORING = {
    "f1_sin_cuenta": make_scorer(f1_score, pos_label=POS_LABEL),
    "recall_sin_cuenta": make_scorer(recall_score, pos_label=POS_LABEL),
    "precision_sin_cuenta": make_scorer(precision_score, pos_label=POS_LABEL, zero_division=0),
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",
}

### 3.7 Baseline: regresión logística balanceada

El **baseline** es el modelo más simple y rápido; sirve de vara de medir: cualquier modelo más complejo tiene que superarlo para justificar su complejidad.

- `LogisticRegression`: modelo lineal que estima la probabilidad como una función logística de una suma ponderada de las variables. **Es interpretable**: cada coeficiente dice cuánto empuja esa variable hacia "tiene cuenta" o "no tiene cuenta".
- `class_weight="balanced"`: da más peso a los errores en la clase minoritaria (sin cuenta), calculado como `n_muestras / (2 · n_clase)`. Sin esto, el modelo tendería a predecir "con cuenta" casi siempre (porque es el 74%) y el recall de excluidos sería pobre.
- `max_iter=1000`: iteraciones máximas del optimizador (por defecto 100 puede no converger con 140 países en one-hot).

Pasos de la celda: crear pipeline → `.fit` solo con train → `evaluar_overfitting` → tabla train vs test.

**Cómo interpretar el resultado:** mira sobre todo `recall_sin_cuenta`, `f1_sin_cuenta` y `roc_auc` en la columna *test*, y que el `gap` sea pequeño. Esos números son el punto de partida que el resto de modelos deben mejorar.

In [5]:
pipe_base = crear_pipeline(LogisticRegression(class_weight="balanced", max_iter=1000,
                                               random_state=RANDOM_STATE))
t0 = time.time()
pipe_base.fit(X_train, y_train)
res_base = evaluar_overfitting(pipe_base, X_train, y_train, X_test, y_test)
res_base["tiempo_ajuste_s"] = time.time() - t0
guardar_json(res_base, TABLES_DIR / "baseline.json")

tabla_base = pd.DataFrame({"train": res_base["train"], "test": res_base["test"]}).drop("umbral")
display(tabla_base.round(4))
print(f"Overfitting (F1_train - F1_test) = {res_base['gap_f1']:.4f} -> "
      f"{'OK (<5%)' if res_base['overfitting_ok'] else 'SUPERA 5%'}")

,train,test
accuracy,0.7773,0.7756
precision_sin_cuenta,0.5507,0.5484
recall_sin_cuenta,0.8173,0.8159
f1_sin_cuenta,0.6580,0.6559
f1_macro,0.7465,0.7447
roc_auc,0.8716,0.8696
pr_auc,0.6911,0.6887
brier,0.1498,0.1510


Overfitting (F1_train - F1_test) = 0.0021 -> OK (<5%)


## 4. Verificar overfitting < 5%

El overfitting ya se mide en `evaluar_overfitting`, pero para el **informe** conviene **mostrar qué es** con un experimento sencillo: entrenamos un árbol de decisión con distintas profundidades.

- Un árbol **poco profundo** (3) es simple: F1 train ≈ F1 test, gap pequeño (quizá algo de *underfitting*).
- Un árbol **sin límite de profundidad** (`None`) memoriza el train: F1 train altísimo, F1 test mucho menor, **gap enorme**.

Esto justifica por qué en todo el notebook **limitamos la complejidad** (`max_depth`, `min_samples_leaf`, regularización L2) y por qué en el tuning solo aceptamos configuraciones con `gap < 0.05`.

La celda construye la tabla profundidad → F1 train / F1 test / gap / ¿cumple?. Si la copias al informe, la frase clave es: *"controlamos el overfitting restringiendo la complejidad del modelo y verificándolo con la diferencia de F1 entre train y test"*.

In [6]:
filas = []
for prof in [3, 6, 10, None]:
    clf = DecisionTreeClassifier(max_depth=prof, class_weight="balanced", random_state=RANDOM_STATE)
    p = crear_pipeline(clf).fit(X_train, y_train)
    r = evaluar_overfitting(p, X_train, y_train, X_test, y_test)
    filas.append({"max_depth": "sin límite" if prof is None else prof,
                  "f1_train": r["train"]["f1_sin_cuenta"], "f1_test": r["test"]["f1_sin_cuenta"],
                  "gap_f1": r["gap_f1"], "cumple_<5%": r["overfitting_ok"]})
tabla_overfit = pd.DataFrame(filas)
tabla_overfit.to_csv(TABLES_DIR / "demo_overfitting_arbol.csv", index=False)
display(tabla_overfit.round(4))

,max_depth,f1_train,f1_test,gap_f1,cumple_<5%
0,3,0.5755,0.5742,0.0013,True
1,6,0.6150,0.6106,0.0045,True
2,10,0.6486,0.6322,0.0164,True
3,sin límite,0.9448,0.5621,0.3827,False


## 5. Comparar modelos

Entrenamos varios algoritmos **con las mismas condiciones** (mismo preprocesador, mismo train, mismo test) y comparamos.

**Catálogo (`catalogo_modelos`)** — qué es cada uno y por qué esos hiperparámetros:

| Modelo | Idea | Controles de complejidad usados |
|---|---|---|
| `logreg` | Lineal, interpretable (el baseline) | — |
| `arbol` | Reglas "si-entonces" encadenadas | `max_depth=8`, `min_samples_leaf=100` |
| `random_forest` | Muchos árboles en paralelo que votan (ensemble *bagging*) | 200 árboles, `max_depth=12`, `min_samples_leaf=50` |
| `hist_gb` | Gradient Boosting: árboles en secuencia, cada uno corrige errores del anterior (ensemble *boosting*) | `learning_rate=0.1`, `max_depth=6`, `l2_regularization=1`, `min_samples_leaf=100` |
| `lightgbm`, `xgboost` | Boosting optimizados (solo si están instalados) | `min_child_samples`/`min_child_weight`, `reg_lambda` |

Notas:
- `min_samples_leaf` alto = cada hoja necesita muchas personas → el modelo no puede memorizar casos individuales (ayuda contra overfitting).
- `class_weight="balanced"` en todos los que lo admiten. XGBoost no lo trae: su desbalance se compensa luego con el **umbral** (§8). Por eso comparamos principalmente por **ROC-AUC** (no depende del umbral) y miramos F1/recall como apoyo.
- `random_state=42` en todos para reproducibilidad. `n_jobs=-1` usa todos los núcleos.

La celda entrena cada modelo, mide train/test y guarda `comparacion_modelos.csv` ordenado por ROC-AUC en test. La columna `overfitting_ok` te dice cuáles cumplen el requisito. **Criterio de elección del modelo final:** el de mayor ROC-AUC entre los que cumplen overfitting < 5%; si hay empate técnico, preferir el más simple/interpretable.

In [7]:
def catalogo_modelos():
    modelos = {
        "logreg": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE),
        "arbol": DecisionTreeClassifier(max_depth=8, min_samples_leaf=100, class_weight="balanced",
                                        random_state=RANDOM_STATE),
        "random_forest": RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=50,
                                                class_weight="balanced", n_jobs=-1,
                                                random_state=RANDOM_STATE),
        "hist_gb": HistGradientBoostingClassifier(max_depth=6, learning_rate=0.1, max_iter=200,
                                                  l2_regularization=1.0, min_samples_leaf=100,
                                                  class_weight="balanced", random_state=RANDOM_STATE),
    }
    if LGBMClassifier is not None:
        modelos["lightgbm"] = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                             min_child_samples=100, reg_lambda=1.0,
                                             class_weight="balanced", random_state=RANDOM_STATE,
                                             n_jobs=-1, verbose=-1)
    if XGBClassifier is not None:
        modelos["xgboost"] = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                                           min_child_weight=20, reg_lambda=2.0, subsample=0.8,
                                           colsample_bytree=0.8, tree_method="hist",
                                           random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss")
    return modelos


filas = []
for nombre, clf in catalogo_modelos().items():
    t0 = time.time()
    pipe = crear_pipeline(clf).fit(X_train, y_train)
    r = evaluar_overfitting(pipe, X_train, y_train, X_test, y_test)
    fila = {"modelo": nombre, "tiempo_s": round(time.time() - t0, 1)}
    for k in ["accuracy", "precision_sin_cuenta", "recall_sin_cuenta", "f1_sin_cuenta", "roc_auc", "pr_auc"]:
        fila[f"{k}_test"] = r["test"][k]
    fila.update(f1_train=r["train"]["f1_sin_cuenta"], gap_f1=r["gap_f1"], overfitting_ok=r["overfitting_ok"])
    filas.append(fila)
    print(f"{nombre:14s} AUC={fila['roc_auc_test']:.4f}  F1={fila['f1_sin_cuenta_test']:.4f}  "
          f"gap={fila['gap_f1']:.4f}  {'OK' if fila['overfitting_ok'] else 'OVERFIT'}")

tabla_comp = pd.DataFrame(filas).sort_values("roc_auc_test", ascending=False).reset_index(drop=True)
tabla_comp.to_csv(TABLES_DIR / "comparacion_modelos.csv", index=False)
display(tabla_comp.round(4))

logreg         AUC=0.8696  F1=0.6559  gap=0.0021  OK
arbol          AUC=0.8325  F1=0.6206  gap=0.0041  OK
random_forest  AUC=0.8516  F1=0.6316  gap=0.0032  OK
hist_gb        AUC=0.8770  F1=0.6675  gap=0.0029  OK
lightgbm       AUC=0.8811  F1=0.6684  gap=0.0070  OK
xgboost        AUC=0.8737  F1=0.6177  gap=0.0095  OK


,modelo,tiempo_s,accuracy_test,precision_sin_cuenta_test,recall_sin_cuenta_test,f1_sin_cuenta_test,roc_auc_test,pr_auc_test,f1_train,gap_f1,overfitting_ok
0,lightgbm,8.9,0.7826,0.5569,0.8358,0.6684,0.8811,0.7163,0.6754,0.0070,True
1,hist_gb,37.2,0.7827,0.5573,0.8321,0.6675,0.8770,0.7083,0.6704,0.0029,True
2,xgboost,15.0,0.8201,0.6975,0.5543,0.6177,0.8737,0.7053,0.6272,0.0095,True
3,logreg,6.6,0.7756,0.5484,0.8159,0.6559,0.8696,0.6887,0.6580,0.0021,True
4,random_forest,23.8,0.7499,0.5146,0.8174,0.6316,0.8516,0.6642,0.6347,0.0032,True
5,arbol,2.8,0.7526,0.5189,0.7718,0.6206,0.8325,0.6014,0.6247,0.0041,True


## 6. Validación cruzada K-Fold estratificada

**Problema del hold-out:** un único split train/test puede ser "suerte" o "mala suerte". La validación cruzada repite el experimento varias veces para saber **cuánto varía** el resultado.

**Cómo funciona `StratifiedKFold(n_splits=5)`:**
1. Divide **solo el train** en 5 trozos (*folds*), **manteniendo la proporción 74/26 en cada uno** (estratificado).
2. Entrena con 4 trozos y valida con el restante; repite 5 veces rotando el trozo de validación.
3. Promedia las 5 puntuaciones y calcula la **desviación estándar** (si es pequeña, el modelo es estable).

Detalles de la celda:
- `shuffle=True, random_state=42` mezcla antes de dividir, de forma reproducible.
- `cross_validate(..., scoring=SCORING, return_train_score=True)` devuelve para cada fold la puntuación en validación **y** en entrenamiento → así calculamos el overfitting también en CV (`gap_f1 = F1_train − F1_val`).
- Como el **preprocesador está dentro del pipeline**, se ajusta de nuevo en cada fold (sin fuga).
- El **test no se toca**.
- Si `MUESTRA` tiene valor, se usa una submuestra para ir rápido.

En el informe reporta `roc_auc_val ± std` y `f1_sin_cuenta_val ± std`: un resultado en CV cercano al de test confirma que el modelo no depende de un split afortunado.

In [8]:
if MUESTRA:
    idx = X_train.sample(min(MUESTRA, len(X_train)), random_state=RANDOM_STATE).index
    Xcv, ycv = X_train.loc[idx], y_train.loc[idx]
else:
    Xcv, ycv = X_train, y_train

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

filas = []
for nombre, clf in catalogo_modelos().items():
    r = cross_validate(crear_pipeline(clf), Xcv, ycv, cv=cv, scoring=SCORING,
                       return_train_score=True, n_jobs=-1)
    fila = {"modelo": nombre}
    for k in SCORING:
        fila[f"{k}_val"] = r[f"test_{k}"].mean()
        fila[f"{k}_val_std"] = r[f"test_{k}"].std()
    fila["f1_train"] = r["train_f1_sin_cuenta"].mean()
    fila["gap_f1"] = fila["f1_train"] - fila["f1_sin_cuenta_val"]
    fila["overfitting_ok"] = fila["gap_f1"] < MAX_GAP
    filas.append(fila)
    print(f"{nombre:14s} AUC={fila['roc_auc_val']:.4f} ± {fila['roc_auc_val_std']:.4f}  "
          f"F1={fila['f1_sin_cuenta_val']:.4f}  gap={fila['gap_f1']:.4f}")

tabla_cv = pd.DataFrame(filas).sort_values("roc_auc_val", ascending=False).reset_index(drop=True)
tabla_cv.to_csv(TABLES_DIR / "validacion_cruzada.csv", index=False)
display(tabla_cv.round(4))

logreg         AUC=0.8705 ± 0.0019  F1=0.6569  gap=0.0014
arbol          AUC=0.8350 ± 0.0031  F1=0.6148  gap=0.0031
random_forest  AUC=0.8519 ± 0.0021  F1=0.6329  gap=0.0019
hist_gb        AUC=0.8772 ± 0.0019  F1=0.6655  gap=0.0059
lightgbm       AUC=0.8808 ± 0.0017  F1=0.6683  gap=0.0087
xgboost        AUC=0.8741 ± 0.0021  F1=0.6211  gap=0.0067


,modelo,f1_sin_cuenta_val,f1_sin_cuenta_val_std,recall_sin_cuenta_val,recall_sin_cuenta_val_std,precision_sin_cuenta_val,precision_sin_cuenta_val_std,roc_auc_val,roc_auc_val_std,accuracy_val,accuracy_val_std,f1_train,gap_f1,overfitting_ok
0,lightgbm,0.6683,0.0021,0.8311,0.0036,0.5589,0.0024,0.8808,0.0017,0.7837,0.0017,0.6770,0.0087,True
1,hist_gb,0.6655,0.0029,0.8235,0.0054,0.5584,0.0031,0.8772,0.0019,0.7829,0.0022,0.6714,0.0059,True
2,xgboost,0.6211,0.0057,0.5547,0.0058,0.7056,0.0060,0.8741,0.0021,0.8226,0.0025,0.6278,0.0067,True
3,logreg,0.6569,0.0034,0.8157,0.0040,0.5498,0.0038,0.8705,0.0019,0.7766,0.0028,0.6583,0.0014,True
4,random_forest,0.6329,0.0018,0.8199,0.0044,0.5154,0.0020,0.8519,0.0021,0.7507,0.0016,0.6348,0.0019,True
5,arbol,0.6148,0.0045,0.8102,0.0293,0.4963,0.0164,0.8350,0.0031,0.7337,0.0139,0.6179,0.0031,True
